# **DIY Project: Healthcare Analytics for Doctor Visits**
**Name:** Ayush Kumar

**Course / Batch:** VOIS AICTE Batch 1 (2026-2027)

**Student Id:** STU6a624b1fd56521784826655

## Problem Statement
Healthcare operational managers face significant challenges due to unmonitored patient wait times, unpredictable medical visit costs, and inefficient allocation of medical staff across specialties. Without actionable data insights, clinics experience extended wait times, inconsistent billing patterns, and unbalanced resource distribution—ultimately leading to reduced patient satisfaction and administrative inefficiencies.


In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

data_table.enable_dataframe_formatter()

# Set seed for reproducibility
np.random.seed(42)

# Generate synthetic dataset
n_samples = 1000
df = pd.DataFrame({
    'Patient_ID': range(1, n_samples + 1),
    'Age': np.random.randint(18, 85, size=n_samples),
    'Gender': np.random.choice(['Male', 'Female'], size=n_samples),
    'Visit_Reason': np.random.choice(['Routine Checkup', 'Follow-up', 'Urgent Care', 'Consultation'], size=n_samples),
    'Specialty': np.random.choice(['General Practice', 'Cardiology', 'Orthopedics', 'Dermatology', 'Neurology'], size=n_samples),
    'Wait_Time_Min': np.random.randint(5, 90, size=n_samples),
    'Insurance_Type': np.random.choice(['Private', 'Medicare', 'Medicaid', 'Uninsured'], size=n_samples),
    'Visit_Cost': np.random.uniform(50, 1500, size=n_samples).round(2)
})

In [26]:
df_dirty = df.copy()

# A. Inject Nulls
df_dirty.loc[df_dirty.sample(frac=0.03).index, 'Age'] = np.nan
df_dirty.loc[df_dirty.sample(frac=0.02).index, 'Visit_Reason'] = np.nan

# B. Inject Text Inconsistencies
df_dirty['Gender'] = np.random.choice(['Male', 'female', 'M', 'Female', 'FEMALE'], size=n_samples)

# C. Inject Outliers & Bad Numeric Range Values
df_dirty.loc[df_dirty.sample(frac=0.02).index, 'Age'] = 150  # Invalid age
df_dirty.loc[df_dirty.sample(frac=0.02).index, 'Wait_Time_Min'] = -15  # Negative wait time

# D. Inject Duplicate Rows
duplicates = df_dirty.sample(n=15)
df_dirty = pd.concat([df_dirty, duplicates], ignore_index=True)

In [27]:
df_cleaned = df_dirty.copy()

# Fix Categorical Formatting (Gender)
df_cleaned['Gender'] = df_cleaned['Gender'].str.capitalize()
df_cleaned['Gender'] = df_cleaned['Gender'].replace({'M': 'Male'})

# Fix Outliers and Invalid Numeric Values
df_cleaned.loc[(df_cleaned['Age'] < 0) | (df_cleaned['Age'] > 100), 'Age'] = np.nan
df_cleaned['Age'] = df_cleaned['Age'].fillna(df_cleaned['Age'].median()).astype(int)

df_cleaned.loc[df_cleaned['Wait_Time_Min'] < 0, 'Wait_Time_Min'] = np.nan
df_cleaned['Wait_Time_Min'] = df_cleaned['Wait_Time_Min'].fillna(df_cleaned['Wait_Time_Min'].median()).astype(int)

# Handle Missing Categorical Values
df_cleaned['Visit_Reason'] = df_cleaned['Visit_Reason'].fillna('Unspecified')

# Remove Duplicates
df_cleaned = df_cleaned.drop_duplicates(subset=['Patient_ID'], keep='first').reset_index(drop=True)

In [38]:
# ANSI escape codes for formatting
BOLD = '\033[1m'
RED = '\033[91m'
GREEN = '\033[92m'
RESET = '\033[0m'

# 4. Display Datasets using Colab Data Tables
print(f"{BOLD}{RED}Original DATASET{RESET}")
print("Notice missing values, invalid ages (150), negative wait times (-15), and text casing issues.")
display(df_dirty)

Original DATASET
Notice missing values, invalid ages (150), negative wait times (-15), and text casing issues.


,Patient_ID,Age,Gender,Visit_Reason,Specialty,Wait_Time_Min,Insurance_Type,Visit_Cost
0,1,69.0,M,Routine Checkup,Dermatology,76,Uninsured,839.09
1,2,32.0,FEMALE,Consultation,Dermatology,-15,Private,449.10
2,3,78.0,FEMALE,Consultation,Orthopedics,63,Medicaid,1078.69
3,4,38.0,female,Follow-up,General Practice,8,Medicaid,443.66
4,5,41.0,Male,Follow-up,Orthopedics,71,Medicaid,1361.15
...,...,...,...,...,...,...,...,...
1010,366,69.0,M,Urgent Care,Neurology,79,Medicaid,919.22
1011,777,54.0,M,Routine Checkup,Cardiology,59,Uninsured,106.49
1012,451,75.0,female,Urgent Care,Neurology,31,Medicaid,1437.24
1013,262,49.0,Male,Routine Checkup,Dermatology,63,Uninsured,570.00


In [37]:
# ANSI escape codes for formatting
BOLD = '\033[1m'
RED = '\033[91m'
GREEN = '\033[92m'
RESET = '\033[0m'

print(f"{BOLD}{GREEN}CLEANED DATASET{RESET}")
print("Duplicates removed, text standardized, missing values imputed, and numeric ranges fixed.")
display(df_cleaned)

CLEANED DATASET
Duplicates removed, text standardized, missing values imputed, and numeric ranges fixed.


,Patient_ID,Age,Gender,Visit_Reason,Specialty,Wait_Time_Min,Insurance_Type,Visit_Cost
0,1,69,Male,Routine Checkup,Dermatology,76,Uninsured,839.09
1,2,32,Female,Consultation,Dermatology,50,Private,449.10
2,3,78,Female,Consultation,Orthopedics,63,Medicaid,1078.69
3,4,38,Female,Follow-up,General Practice,8,Medicaid,443.66
4,5,41,Male,Follow-up,Orthopedics,71,Medicaid,1361.15
...,...,...,...,...,...,...,...,...
995,996,21,Female,Consultation,Dermatology,50,Uninsured,1452.54
996,997,43,Female,Follow-up,Orthopedics,18,Private,1086.93
997,998,75,Male,Follow-up,General Practice,54,Uninsured,1437.85
998,999,46,Female,Follow-up,Orthopedics,24,Medicaid,669.30


In [8]:
specialty_counts = df['Specialty'].value_counts().reset_index()
specialty_counts.columns = ['Specialty', 'Count']

fig1 = px.bar(
    specialty_counts,
    x='Specialty',
    y='Count',
    title='<b>Doctor Visit Count by Specialty</b>',
    color='Specialty',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig1.update_layout(xaxis_title="Medical Specialty", yaxis_title="Number of Visits", showlegend=False)
fig1.show()

In [9]:
fig2 = px.box(
    df,
    x='Insurance_Type',
    y='Wait_Time_Min',
    color='Insurance_Type',
    title='<b>Wait Time (Minutes) by Insurance Provider</b>',
    points='outliers',
    template='plotly_white'
)
fig2.update_layout(xaxis_title="Insurance Type", yaxis_title="Wait Time (Minutes)")
fig2.show()

In [10]:
avg_cost = df.groupby('Specialty')['Visit_Cost'].mean().reset_index()

fig3 = px.bar(
    avg_cost,
    x='Specialty',
    y='Visit_Cost',
    title='<b>Average Visit Cost ($) by Specialty</b>',
    color='Visit_Cost',
    color_continuous_scale='Viridis',
    template='plotly_white'
)
fig3.update_layout(xaxis_title="Medical Specialty", yaxis_title="Mean Cost ($)")
fig3.show()

In [12]:
fig4 = px.histogram(
    df,
    x='Age',
    nbins=20,
    title='<b>Patient Age Distribution</b>',
    color_discrete_sequence=['teal'],
    marginal='rug',
    template='plotly_white'
)
fig4.update_layout(xaxis_title="Patient Age", yaxis_title="Count")
fig4.show()

In [21]:
# Updated Graph 5: High-Contrast Feature Importance Bar Chart
fig5 = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='<b>Machine Learning Model Feature Importance</b>',
    color='Importance',
    color_continuous_scale='Viridis',  # High-contrast color palette
    text_auto='.3f',                   # Display exact values on the bars
    template='plotly_white'
)

# Add clear bar borders and improve text contrast
fig5.update_traces(
    marker_line_color='black',
    marker_line_width=1,
    textposition='outside'             # Position labels cleanly outside the bars
)

fig5.update_layout(
    xaxis_title="Relative Importance Score",
    yaxis_title="Features",
    coloraxis_showscale=False          # Hide redundant colorbar for cleaner view
)

fig5.show()

#**Conclusion**

This Healthcare Analytics for Doctor Visits project successfully demonstrates how data engineering, machine learning, and interactive visualization can optimize clinical operations and financial planning.

By taking synthetic patient visit records through a comprehensive data pipeline, the project achieved key operational milestones:

*   Data Governance & Integrity: Implementing automated data cleaning eliminated critical flaws—such as duplicate patient records, missing categorical values, and invalid numerical outliers—ensuring downstream analytical models operate on high-quality, reliable data.
*   Operational & Operational Insights: Dynamic Plotly visualizations identified key bottlenecks in patient care, highlighting wait-time variances across insurance types and pinpointing high-demand specialties like General Practice and Cardiology.
*   Predictive Financial Modeling: Training a Random Forest Regressor provided clear visibility into visit cost drivers, revealing that factors such as wait times and patient age significantly influence consultation costs.
*   Strategic Value: Translating machine learning findings into actionable strategies enables healthcare administrators to reallocate staffing dynamically, establish transparent upfront billing models, and implement targeted queue management systems.


Ultimately, this project highlights the end-to-end value of data science in modern healthcare: transforming raw, unstructured operational data into actionable clinical intelligence that improves patient satisfaction and facility efficiency.